In [0]:
%sql

MERGE INTO laliga.gold.fact_partidos_stats AS target
USING(
  
    WITH data_partido AS (
    SELECT
      id_partido,
      IdEquipo,
      SUM(Puntos) AS puntos_favor,
      SUM(tiros_2p_totales) + SUM(tiros_3p_totales) AS tiros_campo_totales,
      SUM(libres_totales) AS tiros_libres_totales,
      SUM(ReboteOfensivo) AS rebotes_ofensivos,
      SUM(perdidas) AS perdidas_partido
    FROM laliga.silver.players_stats 
    GROUP BY id_partido, IdEquipo
    ),
    complex_data AS (
    SELECT 
      lo.id_partido,
      lo.IdEquipo,
      lo.puntos_favor,
      --visit.IdEquipo,
      visit.puntos_favor AS puntos_contra,
      lo.tiros_campo_totales,
      lo.tiros_libres_totales,
      lo.rebotes_ofensivos,
      lo.perdidas_partido,
      (lo.tiros_campo_totales + 0.44 * lo.tiros_libres_totales - lo.rebotes_ofensivos + lo.perdidas_partido) AS posesiones, 
      ROUND(lo.puntos_favor / (lo.tiros_campo_totales + 0.44 * lo.tiros_libres_totales - lo.rebotes_ofensivos + lo.perdidas_partido), 2) AS ortg,
      ROUND(visit.puntos_favor / (lo.tiros_campo_totales + 0.44 * lo.tiros_libres_totales - lo.rebotes_ofensivos + lo.perdidas_partido), 2) AS drtg
    FROM data_partido lo
    JOIN data_partido visit ON lo.id_partido = visit.id_partido AND lo.IdEquipo != visit.IdEquipo
    )
    SELECT
      id_partido,
      IdEquipo,
      puntos_favor,
      puntos_contra,
      tiros_campo_totales,
      tiros_libres_totales,
      rebotes_ofensivos,
      perdidas_partido,
      posesiones,
      ortg,
      drtg,
      ROUND(ortg - drtg, 2) AS net_rating,
      'laliga.silver.players_stats' AS _source_table,
      CURRENT_TIMESTAMP() AS _processing_timestamp
    FROM complex_data
) AS source
ON target.id_partido = source.id_partido
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT * FROM laliga.gold.fact_partidos_stats
